# 02 パーティション

**目的**: パーティションが「Spark が並列に処理する単位」であり、その数と大きさが実行時間・偏り・出力ファイル数を決めることを確かめる（60分）。

終わったときに説明できるようになること:

- 読み込んだ DataFrame のパーティション数は何で決まるか
- `repartition` と `coalesce` の違い。`coalesce(1)` の落とし穴
- キーで分割すると偏り（skew）が起きる理由
- 書き出すファイル数は何で決まるか。小さいファイルが大量にできる書き方

In [1]:
import shutil

from pyspark.sql import functions as F

from labkit import (DATA, check_plan, get_spark, large, large_path, list_files, materialize, observe,
                    partition_sizes, partition_summary, timed)

spark = get_spark("02_partitions", {"spark.sql.adaptive.enabled": "false"})

orders = large(spark, "orders")
order_items = large(spark, "order_items")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 20:41:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.4  app=02_partitions  master=local[*]
Spark UI: http://localhost:4040
  spark.sql.adaptive.enabled = false


---

## 実験1: 読み込んだときのパーティション数

1パーティションは1タスクで処理される。8パーティションなら8タスクに分かれ、コアの数だけ同時に走る。

ファイルを読むときのパーティション数は、ファイルの数・サイズと、次の設定で決まる。

- `spark.sql.files.maxPartitionBytes`（既定 128MB）: 1パーティションに詰める上限。これより大きいファイルは分割され、小さいファイルは上限に収まる範囲で1つのパーティションに詰め合わされる
- コア数: 小さいデータでも全コアを使えるように、上限はさらに「全体のサイズ ÷ コア数」あたりまで下げられる

In [2]:
list_files(large_path("orders"))
print("コア数（defaultParallelism）:", spark.sparkContext.defaultParallelism)
print("パーティション数:", orders.rdd.getNumPartitions())

orders: 16 ファイル / 合計 89.1 MB / 平均 5.57 MB
コア数（defaultParallelism）: 8
パーティション数: 8


✏️ **予想: maxPartitionBytes を 8MB に下げて読み直すと、パーティション数はいくつになる？（1ファイルのサイズから考える）**

倍の16になる？

In [6]:
# TODO 1-1: spark.sql.files.maxPartitionBytes を "8m" にしてから orders を読み直し、パーティション数を表示する
#   設定の変更は spark.conf.set(キー, 値)。読み直しは large(spark, "orders")
spark.conf.set("spark.sql.files.maxPartitionBytes","8m")
orders = large(spark, "orders")
print("パーティション数:", orders.rdd.getNumPartitions())

パーティション数: 16


In [7]:
partition_summary(orders, "orders")
partition_sizes(orders).show(30)

[orders] 16 パーティション / 行数 最小 312,499 ・中央値 312,500 ・最大 312,501 （最大÷中央値 = 1.0）
+---------+------+
|partition|  rows|
+---------+------+
|        0|312499|
|        1|312501|
|        2|312500|
|        3|312501|
|        4|312501|
|        5|312499|
|        6|312500|
|        7|312500|
|        8|312500|
|        9|312499|
|       10|312500|
|       11|312500|
|       12|312500|
|       13|312500|
|       14|312500|
|       15|312500|
+---------+------+



✏️ **観察: パーティション数は予想どおりだった？ 最初のセルで見たパーティション数（コア数によって変わる）と比べて、ファイルとパーティションの対応はどう変わった？**

ファイル数とパーティション数が同じになった。これは1ファイルが1パーティションのデータ容量に収まるから

Spark は、ファイルを詰め合わせるとき、1ファイルにつき 4MB の「開くコスト」（`spark.sql.files.openCostInBytes`）を上乗せして計算する。8MB の上限には 5.5MB のファイルが2つ入らないので、1ファイル = 1パーティションになった。

以降の実験は 8MB のまま進める。マシンのコア数によらず、パーティション数を同じ（16）にそろえるためだ。

---

## 実験2: repartition と coalesce

どちらもパーティション数を変えるが、やり方が違う。

In [8]:
# TODO 2-1: orders を 4 パーティションにする。2通りで書く
by_repartition = orders.repartition(4)
by_coalesce = orders.coalesce(4)

In [9]:
by_repartition.explain()
by_coalesce.explain()
check_plan(by_repartition, has=["Exchange RoundRobinPartitioning(4)"])
check_plan(by_coalesce, has=["Coalesce 4"], lacks=["Exchange"])

partition_summary(by_repartition, "repartition(4)")
partition_summary(by_coalesce, "coalesce(4)")

with observe(spark, "repartition(4)"):
    materialize(by_repartition)
with observe(spark, "coalesce(4)"):
    materialize(by_coalesce)

== Physical Plan ==
Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=125]
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#10,customer_id#11,ordered_at#12,status#13,coupon_code#14] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/data/large/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,ordered_at:timestamp,status:string,coupon_code:string>


== Physical Plan ==
Coalesce 4
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#10,customer_id#11,ordered_at#12,status#13,coupon_code#14] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/data/large/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,ordered_at:timestamp,status:string,coupon_code:string>


✅ 計画に `Exchange RoundRobinPartitioning(4)` が ある
✅ 計画に `Coalesce 4` が ある
✅ 計画に `Exchange` が ない
[repartition(4)] 4 パーティ

✏️ **観察: 計画に Exchange（= shuffle）があるのはどちら？ ステージ数、タスク数、shuffle で書き出したサイズ、時間はどう違う？**

ExchangeがあるのはRepartition。
タスク数、ステージ数はRepartitinoが大きく、書き出しが発生しているのもRepartition。処理時間はcoalesceが早い。

- `repartition(n)`: 全行を shuffle して、均等な n 個に配り直す。増やすことも減らすこともできる
- `coalesce(n)`: 今あるパーティションをいくつかずつまとめて n 個に減らす。shuffle しない。増やすことはできず、元のパーティションの大きさが不ぞろいなら、不ぞろいのまま残る

shuffle しない `coalesce` のほうが常に得に見えるが、次の実験のような落とし穴がある。

---

## 実験3: coalesce(1) の落とし穴

「出力を1ファイルにしたい」ときに `coalesce(1)` を付けることがある。下の `heavy` は、明細1行ごとにハッシュを計算する、わざと重くした処理だ。

In [10]:
heavy = (
    order_items
    .withColumn("digest", F.sha2(F.concat_ws("-", "order_id", "line_no", "product_id"), 512))
    .filter(F.col("digest").startswith("00"))
)
print("heavy のパーティション数:", heavy.rdd.getNumPartitions())

heavy のパーティション数: 16


✏️ **予想: heavy.coalesce(1) と heavy.repartition(1) は、どちらが速い？ なぜ？**

（ここに書く）

In [ ]:
# TODO 3-1: heavy を 1 パーティションにまとめて materialize する。coalesce と repartition の両方を測る
with observe(spark, "coalesce(1)"):
    materialize(...)
with observe(spark, "repartition(1)"):
    materialize(...)

✏️ **観察: それぞれのステージ数とタスク数は？ coalesce(1) のとき、重いハッシュ計算は何タスクで実行された？**

（ここに書く）

`coalesce` は shuffle しないので、ステージが分かれない。パーティションを1つにまとめる影響が上流までさかのぼり、ファイルの読み込みも重い計算も、全部が1タスクで直列に実行される。`repartition(1)` は shuffle を挟むので、上流は元のパーティション数のまま並列に走り、最後だけ1つにまとまる。

コア数が多いマシンほど、この差は大きくなる。

---

## 実験4: キーで分割すると偏りが見える

`repartition(n, "列")` は、列の値のハッシュでパーティションを決める。同じ値の行は必ず同じパーティションに入る。groupBy や join の shuffle も、これと同じ分け方をする。

In [ ]:
# TODO 4-1: orders を customer_id をキーにして 200 パーティションに分ける
by_customer = ...

In [ ]:
check_plan(by_customer, has=["Exchange hashpartitioning(customer_id"])
partition_summary(by_customer, "customer_id で 200 分割")
partition_sizes(by_customer).orderBy(F.desc("rows")).show(5)

✏️ **観察: 最大のパーティションは、中央値の何倍の行数？ なぜそのパーティションだけ大きい？（00_setup で見た顧客ごとの注文数を思い出す）**

（ここに書く）

1つのタスクだけが他の何十倍ものデータを処理することになり、そのタスクが終わるまでステージ全体が終わらない。これが skew（データの偏り）で、Spark のジョブが遅い・メモリ不足で落ちる原因の定番になっている。対処は `04_joins` で扱う。

---

## 実験5: 書き出すファイル数

書き出すとき、Spark は**1パーティションにつき1ファイル**を作る。`partitionBy("列")` を付けると、各パーティションが、その中に含まれる列の値ごとに1ファイルを作る。

In [ ]:
# TODO 5-1: orders に order_month 列（ordered_at を "yyyy-MM" 形式にした文字列）を足す
#   使うもの: withColumn / F.date_format
with_month = ...

✏️ **予想: with_month をこのまま partitionBy("order_month") で書き出すと、ファイルはいくつできる？（パーティション数 × 月の数 で考える）**

（ここに書く）

In [ ]:
out = str(DATA / "tmp" / "02_partitions")

with_month.write.mode("overwrite").partitionBy("order_month").parquet(out + "/as_is")
as_is = list_files(out + "/as_is")

In [ ]:
# TODO 5-2: 月ごとに1ファイルになるように書き出したい。
#   書き出す前に、with_month を「同じ月の行が1つのパーティションに集まる」ように分け直す
regrouped = ...

In [ ]:
regrouped.write.mode("overwrite").partitionBy("order_month").parquet(out + "/one_file_per_month")

In [ ]:
compact = list_files(out + "/one_file_per_month")
n_months = with_month.select("order_month").distinct().count()
print("✅ 月ごとに1ファイル" if compact["files"] == n_months else f"❌ {n_months} ファイルになるはず")

with observe(spark, "小さいファイルが多いほうを読む"):
    materialize(spark.read.parquet(out + "/as_is"))
with observe(spark, "月ごとに1ファイルのほうを読む"):
    materialize(spark.read.parquet(out + "/one_file_per_month"))

✏️ **観察: ファイル数は何倍違った？ 合計サイズは？ 読むときのタスク数と時間は？**

（ここに書く）

小さいファイルが大量にあると、ファイルを開くコストと、ファイル一覧を管理するコストが、中身を読むコストを上回るようになる（small files problem）。追記を繰り返すテーブルは、放っておくと必ずこうなる。Delta Lake の `OPTIMIZE` は、これを後からまとめ直すコマンドだ。

In [ ]:
shutil.rmtree(DATA / "tmp", ignore_errors=True)

---

## まとめ

✏️ **このノートブックで分かったことを、自分の言葉で3行にまとめる**

（ここに書く）

### 説明できるか確かめる

1. パーティション数が少なすぎると何が困る？ 多すぎると何が困る？
2. `repartition` と `coalesce` の違いを、shuffle の有無から説明できる？ それぞれどんなときに使う？
3. 出力を1ファイルにしたいとき、`coalesce(1)` で問題ないのはどんな場合？
4. groupBy や join で、一部のタスクだけ極端に遅くなるのはなぜ？
5. 日次で追記しているテーブルのファイル数が増え続けている。何が起きていて、どう直す？

In [ ]:
spark.stop()